**Setup, Drive Mounting and Image Extraction**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import pandas as pd
import numpy as np
import os
import zipfile
import shutil
from tqdm import tqdm
from collections import Counter
import json
import random
from sklearn.metrics import accuracy_score, classification_report

## Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

##PATHS & EXTRACTION
BACKUP_DIR = '/content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1'
ZIP_PATH = '/content/drive/MyDrive/Thesis-work/ISIC_2019_Training_Input.zip'
UNLABELED_DIR = 'data/isic2019/ISIC_2019_Training_Input'
LABELED_DIR = 'data/labeled_real'

print("Extracting ISIC 2019 images...")

os.makedirs('data/isic2019', exist_ok=True)
os.makedirs(LABELED_DIR, exist_ok=True)

if not os.path.exists(UNLABELED_DIR) or len(os.listdir(UNLABELED_DIR)) < 1000:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall('data/isic2019/')
    print(f"Extracted to data/isic2019/")
else:
    print("Already extracted")

if os.path.exists(UNLABELED_DIR):
    unlabeled_count = len([f for f in os.listdir(UNLABELED_DIR) if f.endswith('.jpg')])
    print(f"Unlabeled images: {unlabeled_count}")
else:
    print("Extraction failed")

## Verify extraction
if os.path.exists(UNLABELED_DIR):
    unlabeled_count = len([f for f in os.listdir(UNLABELED_DIR) if f.endswith('.jpg')])
    print(f" Unlabeled images: {unlabeled_count}")
else:
    print(" Extraction failed - check ZIP path")
    print(f"ZIP exists: {os.path.exists(ZIP_PATH)}")
    ## List what's in data/isic2019/
    if os.path.exists('data/isic2019'):
        print("Contents:", os.listdir('data/isic2019'))

## FILTERING 691 LABELED IMAGES

print("\nFiltering to labeled images...")

train_df = pd.read_csv(f'{BACKUP_DIR}/expA_train.csv')
val_df = pd.read_csv(f'{BACKUP_DIR}/expA_val.csv')
test_df = pd.read_csv(f'{BACKUP_DIR}/expA_test.csv')
all_images = pd.concat([train_df, val_df, test_df])['image'].unique()

found = 0
for img_id in all_images:
    src = f'{UNLABELED_DIR}/{img_id}.jpg'
    dst = f'{LABELED_DIR}/{img_id}.jpg'
    if os.path.exists(src):
        shutil.copy(src, dst)
        found += 1

print(f"Copied {found}/{len(all_images)} labeled images")

if found > 0:
    sample = os.listdir(LABELED_DIR)[0]
    img = Image.open(f'{LABELED_DIR}/{sample}')
    print(f"Sample: {sample}, size: {img.size}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
Extracting ISIC 2019 images...
Extracted to data/isic2019/
Unlabeled images: 25331
 Unlabeled images: 25331

Filtering to labeled images...
Copied 691/691 labeled images
Sample: ISIC_0033602.jpg, size: (600, 450)


**Loading Unlabeled Data Pool**

In [4]:
## Loading unlabeled metadata
unlabeled_df = pd.read_csv(f'{BACKUP_DIR}/unlabeled_pool_24k.csv')
unlabeled_ids = unlabeled_df['image'].tolist()

## Filter to only images that actually exist
available_ids = []
missing = []
for img_id in unlabeled_ids:
    if os.path.exists(f'{UNLABELED_DIR}/{img_id}.jpg'):
        available_ids.append(img_id)
    else:
        missing.append(img_id)

print(f"\nTotal in metadata: {len(unlabeled_ids)}")
print(f"Available on disk: {len(available_ids)}")
print(f"Missing: {len(missing)}")

## Use available images (might be slightly less than 24K)
unlabeled_sample = available_ids
print(f"\n Using {len(unlabeled_sample)} real unlabeled images for SSL")

print(f"\nLabeled splits: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
print(f"Train distribution:\n{train_df['label'].value_counts()}")


Total in metadata: 24000
Available on disk: 24000
Missing: 0

 Using 24000 real unlabeled images for SSL

Labeled splits: Train=483, Val=104, Test=104
Train distribution:
label
NV     349
BKL     99
MEL     35
Name: count, dtype: int64


#**Data types from the source**

In [5]:
import os

# Check data/isic2019/ structure
print("Contents of data/isic2019/:")
for item in os.listdir('data/isic2019/'):
    print(f"  {item}")
    # If it's a folder, check inside
    path = f'data/isic2019/{item}'
    if os.path.isdir(path):
        jpg_count = len([f for f in os.listdir(path) if f.endswith('.jpg')])
        print(f"    JPG images: {jpg_count}")

# Also check if there's a deeper nesting
for root, dirs, files in os.walk('data/isic2019/'):
    jpg_files = [f for f in files if f.endswith('.jpg')]
    if jpg_files:
        print(f"\nFound {len(jpg_files)} images in: {root}")
        print(f"Sample: {jpg_files[:3]}")
        break  # Only show first location with images

Contents of data/isic2019/:
  ISIC_2019_Training_Input
    JPG images: 25331

Found 25331 images in: data/isic2019/ISIC_2019_Training_Input
Sample: ['ISIC_0032970.jpg', 'ISIC_0027373.jpg', 'ISIC_0061770.jpg']


**SimCLR Components Assign**

In [6]:
## SimCLR Transform (two views)
class SimCLRTransform:
    def __init__(self, size=224):
        self.transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    def __call__(self, x):
        # Return two different random augmentations of the same image
        return self.transform(x), self.transform(x)

## Dataset
class UnlabeledDataset(Dataset):
    def __init__(self, image_ids, image_dir, transform):
        self.image_ids = image_ids
        self.image_dir = image_dir
        self.transform = transform
    def __len__(self):
        return len(self.image_ids)
    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        img = Image.open(f'{self.image_dir}/{img_id}.jpg').convert('RGB')
        return self.transform(img)

## Encoder with Batch Normalization for stability
class SimpleEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d(1), nn.Flatten()
        )
    def forward(self, x):
        return self.features(x)

## SimCLR Model
class SimCLR(nn.Module):
    def __init__(self, encoder, projection_dim=64):
        super().__init__()
        self.encoder = encoder
        self.projector = nn.Sequential(
            nn.Linear(128, 128), nn.ReLU(), nn.Linear(128, projection_dim)
        )
    def forward(self, x):
        h = self.encoder(x)
        z = F.normalize(self.projector(h), dim=1)
        return h, z

## NT-Xent Loss with stability epsilon
def nt_xent_loss(z_i, z_j, temperature=0.5):
    batch_size = z_i.shape[0]
    z = torch.cat([z_i, z_j], dim=0)
    sim_matrix = torch.mm(z, z.t()) / temperature

    # Mask out self-similarity
    mask = torch.eye(2 * batch_size, device=z.device).bool()
    sim_matrix = sim_matrix.masked_fill(mask, -9e15)

    pos_sim = torch.cat([
        torch.diag(sim_matrix, batch_size),
        torch.diag(sim_matrix, -batch_size)
    ])

    # Numerically stable softmax/logsumexp approach
    log_sum_exp = torch.logsumexp(sim_matrix, dim=1)
    loss = - (pos_sim - log_sum_exp)

    return loss.mean()

print("SimCLR components updated: guaranteed distinct random views.")

SimCLR components updated: guaranteed distinct random views.


#**SSL Training**

In [7]:
## Initialize
encoder = SimpleEncoder()
simclr_model = SimCLR(encoder).to(device)

simclr_transform = SimCLRTransform()
unlabeled_dataset = UnlabeledDataset(unlabeled_sample, UNLABELED_DIR, simclr_transform)
unlabeled_loader = DataLoader(
    unlabeled_dataset,
    batch_size=64,
    shuffle=True,
    drop_last=True,
    num_workers=0 # Increased workers for performance
)

# Lower learning rate slightly for stability
optimizer = torch.optim.Adam(simclr_model.parameters(), lr=0.0003)

print(f"\n{'='*36}")
print(f"Training SimCLR on {len(unlabeled_sample)} REAL images")
print(f"Device: {'GPU (' + torch.cuda.get_device_name(0) + ')' if torch.cuda.is_available() else 'CPU'}")
print(f"{'='*36}\n")

epochs = 20
checkpoint_dir = f'{BACKUP_DIR}/ssl_checkpoints_real_24k'
os.makedirs(checkpoint_dir, exist_ok=True)

for epoch in range(epochs):
    simclr_model.train()
    total_loss = 0
    pbar = tqdm(unlabeled_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for view1, view2 in pbar:
        view1, view2 = view1.to(device), view2.to(device)
        _, z1 = simclr_model(view1)
        _, z2 = simclr_model(view2)

        loss = nt_xent_loss(z1, z2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_loss = total_loss / len(unlabeled_loader)
    print(f"Epoch {epoch+1}/{epochs}, Avg Loss: {avg_loss:.4f}")

    if (epoch + 1) % 5 == 0:
        ckpt_path = f'{checkpoint_dir}/ssl_real_epoch_{epoch+1}.pth'
        torch.save(encoder.state_dict(), ckpt_path)
        print(f"Checkpoint saved: {ckpt_path}")

final_path = f'{BACKUP_DIR}/ssl_encoder_real_{len(unlabeled_sample)}.pth'
torch.save(encoder.state_dict(), final_path)
print(f"\n Final encoder saved: {final_path}")


Training SimCLR on 24000 REAL images
Device: GPU (Tesla T4)



Epoch 1/20: 100%|██████████| 375/375 [14:48<00:00,  2.37s/it, loss=3.5269]


Epoch 1/20, Avg Loss: 3.7100


Epoch 2/20: 100%|██████████| 375/375 [14:34<00:00,  2.33s/it, loss=3.4483]


Epoch 2/20, Avg Loss: 3.4820


...

Epoch 18/20, Avg Loss: 3.1776


Epoch 19/20: 100%|██████████| 375/375 [14:38<00:00,  2.34s/it, loss=3.1526]


Epoch 19/20, Avg Loss: 3.1726


Epoch 20/20: 100%|██████████| 375/375 [14:44<00:00,  2.36s/it, loss=3.1509]


Epoch 20/20, Avg Loss: 3.1662
Checkpoint saved: /content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1/ssl_checkpoints_real_24k/ssl_real_epoch_20.pth

 Final encoder saved: /content/drive/MyDrive/Thesis-work/Backups/thesis_backup_day1/ssl_encoder_real_24000.pth


**Fine-tuning Real Data with SSL Encoder**

In [8]:
## Transforms
normal_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

mel_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(0.4, 0.4, 0.3, 0.1),
    transforms.RandomAffine(15, translate=(0.1, 0.1), scale=(0.85, 1.15)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

class AugmentedDataset(Dataset):
    def __init__(self, df, mel_multiplier=5, transform_normal=None, transform_mel=None):
        self.df = df
        self.classes = sorted(df['label'].unique())
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}
        self.transform_normal = transform_normal
        self.transform_mel = transform_mel

        mel_rows = df[df['label'] == 'MEL']
        other_rows = df[df['label'] != 'MEL']
        mel_expanded = pd.concat([mel_rows] * mel_multiplier, ignore_index=True)
        self.expanded_df = pd.concat([mel_expanded, other_rows], ignore_index=True)\
                          .sample(frac=1, random_state=42).reset_index(drop=True)

    def __len__(self):
        return len(self.expanded_df)

    def __getitem__(self, idx):
        row = self.expanded_df.iloc[idx]
        img = Image.open(f"{LABELED_DIR}/{row['image']}.jpg").convert('RGB')
        label = self.class_to_idx[row['label']]
        if row['label'] == 'MEL' and self.transform_mel:
            img = self.transform_mel(img)
        elif self.transform_normal:
            img = self.transform_normal(img)
        return img, label

## Create datasets
train_ds = AugmentedDataset(train_df, 5, normal_transform, mel_transform)
val_ds = AugmentedDataset(val_df, 1, test_transform, test_transform)
test_ds = AugmentedDataset(test_df, 1, test_transform, test_transform)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)
test_loader = DataLoader(test_ds, batch_size=16)

print(f"\nFine-tuning datasets: Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}")

## Model
class SSLClassifier(nn.Module):
    def __init__(self, encoder, num_classes):
        super().__init__()
        self.encoder = encoder
        self.classifier = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.encoder(x))

## Load SSL encoder
encoder = SimpleEncoder()
encoder.load_state_dict(torch.load(final_path, map_location=device))
print(f"Loaded SSL encoder (trained on REAL {len(unlabeled_sample)} images)")

## Unfreeze all
for param in encoder.parameters():
    param.requires_grad = True

model = SSLClassifier(encoder, len(train_ds.classes)).to(device)

## Loss
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        return ((1 - pt) ** self.gamma * ce_loss).mean()

weight_dict = {'MEL': 4.0, 'BKL': 2.0, 'NV': 1.0}
weights_list = [weight_dict[cls] for cls in train_ds.classes]
weights = torch.tensor(weights_list, dtype=torch.float32).to(device)
criterion = FocalLoss(alpha=weights, gamma=1.5)
optimizer = optim.Adam(model.parameters(), lr=0.0001)

print(f"Weights: {dict(zip(train_ds.classes, weights_list))}")


Fine-tuning datasets: Train=623, Val=104, Test=104
Loaded SSL encoder (trained on REAL 24000 images)
Weights: {'BKL': 2.0, 'MEL': 4.0, 'NV': 1.0}


#**Final Training and Evaluation**

In [11]:
from sklearn.metrics import accuracy_score, classification_report

def train_epoch(model, loader):
    model.train()
    total_loss, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        correct += (outputs.argmax(1) == labels).sum().item()
    return total_loss / len(loader), correct / len(loader.dataset)

def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = model(images)
            preds = outputs.argmax(1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return accuracy_score(all_labels, all_preds), all_labels, all_preds

##Train
print(f"\n{'='*36}")
print(f"Fine-tuning SSL (real {len(unlabeled_sample)}) + Augmentation")
print(f"{'='*36}")

epochs = 30
best_val = 0
patience = 7
epochs_no_improve = 0

for epoch in range(epochs):
    train_loss, train_acc = train_epoch(model, train_loader)
    val_acc, _, _ = evaluate(model, val_loader)

    if val_acc > best_val:
        best_val = val_acc
        torch.save(model.state_dict(), f'best_ssl_real_{len(unlabeled_sample)}.pth')
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if (epoch + 1) % 3 == 0:
        print(f"Epoch {epoch+1}: Train={train_acc:.3f}, Val={val_acc:.3f}")

    if epochs_no_improve >= patience:
        print(f"Early stopping at epoch {epoch+1}")
        break

## Evaluate
model.load_state_dict(torch.load(f'best_ssl_real_{len(unlabeled_sample)}.pth', map_location=device))
test_acc, true_labels, pred_labels = evaluate(model, test_loader)

print("\n" + "="*36)
print(f"RESULTS: SSL (REAL {len(unlabeled_sample)}) + AUGMENTATION")
print("="*36)
print(f"Test Accuracy: {test_acc:.3f}")

report = classification_report(true_labels, pred_labels,
                                target_names=train_ds.classes, output_dict=True)
print(classification_report(true_labels, pred_labels, target_names=train_ds.classes))

mel_recall = report['MEL']['recall']
print(f"\n>>> MEL Recall: {mel_recall:.1%} <<<")
print(f"Predictions: {Counter(pred_labels)}")

##Saving Results
results = {
    'model': f'SSL_Real_{len(unlabeled_sample)}',
    'ssl_data': f'{len(unlabeled_sample)} real unlabeled ISIC',
    'augmentation': 'MEL 5x + heavy transforms',
    'test_accuracy': test_acc,
    'mel_recall': mel_recall,
    'report': report
}
with open(f'ssl_real_{len(unlabeled_sample)}_results.json', 'w') as f:
    json.dump(results, f, indent=2)

torch.save(model.state_dict(), f'{BACKUP_DIR}/ssl_real_{len(unlabeled_sample)}_finetuned.pth')
print(f"\n Saved to Drive")


Fine-tuning SSL (real 24000) + Augmentation
Epoch 3: Train=0.581, Val=0.606
Epoch 6: Train=0.660, Val=0.644
Epoch 9: Train=0.682, Val=0.654
Epoch 12: Train=0.698, Val=0.625
Early stopping at epoch 14

RESULTS: SSL (REAL 24000) + AUGMENTATION
Test Accuracy: 0.673
              precision    recall  f1-score   support

         BKL       1.00      0.14      0.25        21
         MEL       0.21      0.50      0.30         8
          NV       0.77      0.84      0.80        75

    accuracy                           0.67       104
   macro avg       0.66      0.49      0.45       104
weighted avg       0.77      0.67      0.65       104


>>> MEL Recall: 50.0% <<<
Predictions: Counter({np.int64(2): 82, np.int64(1): 19, np.int64(0): 3})

 Saved to Drive
